# 💼 The Analyst's Notebook · Part 12
### Boosting, and the report as a whole

After Part 11 the risk report holds two results, each with a flexible challenger behind it. The volatility forecast is still the one-column linear regression chosen in Part 5: 0.00412 on the test block and 0.00755 on the folds, where ridge on twenty columns scored 0.00460 and 0.00761 and a forest tuned on the folds 0.00462 and 0.00757. The jump warning is still Part 9's one-column logistic regression, with an AUC of 0.889, run at a threshold of 0.25.

This part does two things. It first gives boosting the question the report was built on, with the number of trees chosen on 2022, and reads the boosted model through its importance. Then, because this is the last part of the case, it sets every model the report has fitted since Part 4 side by side: on the same rows and both yardsticks, across the desk, and on the days that went wrong. It ends with the report as it would go out on the last day of the data.

**Course site:** [the lecture](https://theill95.github.io/mlfin-2026/session_12/session_12.html) · [the exercises, in Colab](https://colab.research.google.com/github/theill95/mlfin-2026/blob/main/session_12/session_12_exercises.ipynb) · [the cheatsheet](https://theill95.github.io/mlfin-2026/cheatsheet.html) · [all sessions](https://theill95.github.io/mlfin-2026/)

## How to work through this

- Run the **quick load** cell first. It brings back what the risk report holds after Part 11 and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.
- Q9 refits sixteen models on the training block and the folds, and takes about a minute on Colab.

**A note on units.** The lecture worked in percent, on the index table. This notebook keeps the plain decimals of Parts 1 to 11 and stays on Apple, so every number compares directly with the report's. Boosted trees, like single trees, do not mind the units.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what the risk report holds after Part 11. Run it and read what it prints.

In [ ]:
# The numbers in this notebook come from XGBoost 2.1.4, the version the lecture runs. Colab
# ships a newer XGBoost that grows slightly different trees, so there the first run installs
# 2.1.4 (about 20 seconds, once per Colab session).
import sys
import importlib.metadata
try:
    xgboost_version = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    xgboost_version = None
if xgboost_version != "2.1.4":
    if "google.colab" in sys.modules:
        import subprocess
        print("Installing XGBoost 2.1.4 for this Colab session...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost==2.1.4"], check=True)
    else:
        print(f"Note: you have XGBoost {xgboost_version}; the numbers here come from 2.1.4, "
              "which the setup guide's install line gives you.")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor, XGBClassifier             # the setup guide's install line includes it

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as in Part 6, as a plain number."""
    return float(np.sqrt(mean_squared_error(actual, predicted)))


# The whole universe: eleven instruments, 2015 to 2024, returns in plain decimals
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

folds = TimeSeriesSplit(n_splits=5)

# --- What the risk report holds after Part 11, on Apple ---
part11_split = "by date: train to 2022-12-31, test from 2023-01-01"
part5_target = "sd of daily returns over the next 20 trading days"

# the forecast: Part 6's linear models and Part 11's trees, on the test block and the folds
part6_one_rmse = 0.00412        # linear regression on vol_20d
part6_one_folds = 0.00755
part6_ridge_rmse = 0.00460      # ridge on all twenty columns
part6_ridge_folds = 0.00761
part6_ridge_alpha = 1000         # the settings the folds chose in Part 6
part6_lasso_alpha = 0.001
part6_enet = {'alpha': 0.001, 'l1_ratio': 0.9}
part6_desk = {                  # (ridge, one column) on each instrument's test block
    'AAPL': (0.00460, 0.00412),
    'DIS': (0.00559, 0.00645),
    'JNJ': (0.00307, 0.00314),
    'JPM': (0.00559, 0.00565),
    'KO': (0.00281, 0.00272),
    'MSFT': (0.00349, 0.00350),
    'NVDA': (0.01112, 0.01137),
    'PG': (0.00284, 0.00283),
    'SPY': (0.00218, 0.00237),
    'WMT': (0.00401, 0.00408),
    'XOM': (0.00327, 0.00322)
}
part11_stump_rmse = 0.00416     # a stump on vol_20d
part11_forest_settings = {'max_features': 5, 'min_samples_leaf': 100}   # chosen on the folds
part11_forest_rmse = 0.00462    # the tuned forest on all twenty columns
part11_forest_folds = 0.00757
part11_desk_forest = {          # the same forest on each instrument's test block
    'AAPL': 0.00462,
    'DIS': 0.00543,
    'JNJ': 0.00346,
    'JPM': 0.00573,
    'KO': 0.00287,
    'MSFT': 0.00338,
    'NVDA': 0.01094,
    'PG': 0.00288,
    'SPY': 0.00245,
    'WMT': 0.00396,
    'XOM': 0.00323
}

# the warning: Part 9's label and models, and Part 11's forest on it (AUC)
part9_label = "jump: vol_next more than 1.5 times vol_20d"
part9_auc = 0.8892              # logistic regression on vol_20d, the test block
part9_folds_auc = 0.7923        # the same model on the folds
part9_wide_auc = 0.7458         # twenty columns, C chosen on the folds
part9_wide_folds_auc = 0.7547
part9_knn_auc = 0.8378          # k-nearest neighbours on vol_20d, k chosen on the folds
part9_knn_folds_auc = 0.7586
part9_threshold = 0.25          # chosen from the desk's costs: a miss 5, a false alarm 1
part11_jump_forest_auc = 0.8686        # Part 11's tuned forest on the jump label
part11_jump_forest_folds_auc = 0.6824  # the same forest on the folds

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("After Part 11 the risk report holds, on Apple:")
print(f"  a forecast: one column, RMSE {part6_one_rmse:.5f} on the test block and {part6_one_folds:.5f} on the folds")
print(f"              (ridge {part6_ridge_rmse:.5f} and {part6_ridge_folds:.5f}; the tuned forest "
      f"{part11_forest_rmse:.5f} and {part11_forest_folds:.5f})")
print(f"  a warning : one column, AUC {part9_auc:.3f} on the test block and {part9_folds_auc:.3f} on the folds,")
print(f"              run at a threshold of {part9_threshold}")
print()
print("This is the last part. Boosting first, then every model side by side.")

---

### Q1 · Where the report stands

Write `wide_table(ticker, keep_unknown=False)`. With the default it returns the table Parts 6 to 11 worked on, as you wrote it in Part 11: the six volatility windows, the three return windows, `up_20d`, the 20-day volatility of every other instrument, the target `vol_next`, and incomplete rows dropped. With `keep_unknown=True` it also keeps the last days of the data, whose features are known and whose `vol_next` is not yet, by dropping only rows with a missing feature. Build Apple's table both ways, as `table` and `latest`, store the twenty feature names in `columns`, and split `table` at the end of 2022. Then refit the report's forecast as `one_model` and check its test RMSE, `one_rmse`, against `part6_one_rmse`.

In [ ]:
def wide_table(ticker, keep_unknown=False):
    """..."""
    ...


table = ...
latest = ...
columns = ...
train = ...
test = ...

one_model = ...
...
one_rmse = ...

print('days whose target is not known yet:', ...)
print(one_rmse)
print('matches the report:', ...)

<details>
<summary>💡 Hint 1</summary>

Build `frame` as in Part 11. Before the last line, `if keep_unknown:` returns `frame.dropna(subset=list(frame.columns[:-1]))`, which only looks at the feature columns; otherwise `frame.dropna()` as before.

</details>

<details>
<summary>💡 Hint 2</summary>

`len(latest) - len(table)` counts the extra days, and the check is `abs(one_rmse - part6_one_rmse) < 0.00001`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def wide_table(ticker, keep_unknown=False):
    """The report's twenty columns and the target for one instrument, in plain decimals.
    keep_unknown=True keeps the last days, whose target is not known yet."""
    frame = pd.DataFrame()
    for w in [5, 10, 20, 40, 60, 120]:
        frame['vol_' + str(w) + 'd'] = rets[ticker].rolling(w).std()
    for w in [5, 20, 60]:
        frame['ret_' + str(w) + 'd'] = rets[ticker].rolling(w).mean()
    frame['up_20d'] = (rets[ticker] > 0).rolling(20).mean()
    for t in TICKERS:
        if t != ticker:
            frame[t + '_vol'] = rets[t].rolling(20).std()
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    if keep_unknown:
        return frame.dropna(subset=list(frame.columns[:-1]))
    return frame.dropna()


table = wide_table('AAPL')
latest = wide_table('AAPL', keep_unknown=True)
columns = list(table.columns[:-1])
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

one_model = LinearRegression()
one_model.fit(train[['vol_20d']], train['vol_next'])
one_rmse = rmse(test['vol_next'], one_model.predict(test[['vol_20d']]))

print('days whose target is not known yet:', len(latest) - len(table))
print(one_rmse)
print('matches the report:', abs(one_rmse - part6_one_rmse) < 0.00001)
```

20 days, 3 to 31 December 2024: the data ends before their next twenty trading days do. Then 0.00412 and `True`, on 1,894 training days and 482 test days. Every model in this part is held to the forecast's two scores, the test block and the folds. Q14 returns to `latest` for the forecast those last 20 days are waiting for.

</details>

---

### Q2 · The first stump is Part 11's

Part 11's stump on `vol_20d` scored 0.00416 with two numbers. Boosting starts from the training average and adds a tenth of a stump fitted to what is left. Refit Part 11's stump as `stump`, and fit `GradientBoostingRegressor` with 100 stumps at a learning rate of 0.1 on `vol_20d` as `boost_one`. Check that its first stage is the average plus a tenth of the stump's distance from it, $\hat f_1 = \bar y + 0.1\,(\text{stump} - \bar y)$. Then print the training RMSE after 1, 10 and 100 stumps.

In [ ]:
stump = ...
...
boost_one = ...
...

average = ...
stages = ...
check = ...

print('first stage = average + 0.1 x (stump - average):', check)
print(...)

<details>
<summary>💡 Hint 1</summary>

`stages = list(boost_one.staged_predict(train[['vol_20d']]))`, so `stages[0]` is the first stage. The check is `np.allclose(stages[0], average + 0.1 * (stump.predict(train[['vol_20d']]) - average))`.

</details>

<details>
<summary>💡 Hint 2</summary>

`[rmse(train['vol_next'], stages[m - 1]) for m in [1, 10, 100]]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump = DecisionTreeRegressor(max_depth=1, random_state=0)
stump.fit(train[['vol_20d']], train['vol_next'])
boost_one = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
boost_one.fit(train[['vol_20d']], train['vol_next'])

average = train['vol_next'].mean()
stages = list(boost_one.staged_predict(train[['vol_20d']]))
check = np.allclose(stages[0], average + 0.1 * (stump.predict(train[['vol_20d']]) - average))

print('first stage = average + 0.1 x (stump - average):', check)
print([round(rmse(train['vol_next'], stages[m - 1]), 5) for m in [1, 10, 100]])
```

`True`, and 0.00816, 0.00748 and 0.00704. A stump fitted to y − ȳ cuts where a stump fitted to y cuts, at 0.0175, and forecasts each side's mean minus ȳ, so boosting's first step is Part 11's stump, a tenth of the way. Every later stump is fitted to what the earlier ones left, and the training error falls with each. How many to keep is a setting, and Q3 chooses it without the test block.

</details>

---

### Q3 · How many stumps, chosen on 2022

Split the training days into `fit_rows`, 2015 to 2021, and `stop_rows`, 2022. Fit 1,000 stumps at a learning rate of 0.1 on `vol_20d` in `fit_rows` as `boost_stop`, score every stage on 2022 into a list `stop_scores`, and keep the number of stumps with the lowest 2022 RMSE as `best_m`. Draw the 2022 RMSE against the number of stumps with `best_m` marked. Then open the test block once, for the model with `best_m` stumps, as `stopped_rmse`.

In [ ]:
fit_rows = ...
stop_rows = ...
boost_stop = ...
...
stop_scores = ...
best_m = ...

fig, ax = plt.subplots(figsize=(9, 3))
...
...
plt.show()

stopped_rmse = ...
print(best_m, stopped_rmse, ' one column:', one_rmse)

<details>
<summary>💡 Hint 1</summary>

`stop_scores = [rmse(stop_rows['vol_next'], f) for f in boost_stop.staged_predict(stop_rows[['vol_20d']])]` and `best_m = int(np.argmin(stop_scores)) + 1`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(range(1, 1001), stop_scores)` and `ax.scatter([best_m], [min(stop_scores)])`. The test forecast with `best_m` stumps is `list(boost_stop.staged_predict(test[['vol_20d']]))[best_m - 1]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_rows = train.loc[:'2021-12-31']
stop_rows = train.loc['2022-01-01':]
boost_stop = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=1, random_state=0)
boost_stop.fit(fit_rows[['vol_20d']], fit_rows['vol_next'])
stop_scores = [rmse(stop_rows['vol_next'], f) for f in boost_stop.staged_predict(stop_rows[['vol_20d']])]
best_m = int(np.argmin(stop_scores)) + 1

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(range(1, 1001), stop_scores, color='#1c5cab')
ax.scatter([best_m], [min(stop_scores)], s=60, color='#b3402f', zorder=3)
ax.set_xlabel('stumps, fitted on 2015 to 2021')
ax.set_ylabel('RMSE on 2022')
ax.set_title(f'Boosted stumps on vol_20d: the lowest 2022 RMSE at {best_m}', loc='left')
plt.show()

stopped_rmse = rmse(test['vol_next'], list(boost_stop.staged_predict(test[['vol_20d']]))[best_m - 1])
print(best_m, stopped_rmse, ' one column:', one_rmse)
```

46 stumps, at 0.00636 on 2022, against 0.00736 after one stump and 0.00659 after 1,000; anything from 45 to 89 stumps comes within 0.00001 of the best. On the test block the 46 stumps score 0.00375, the lowest number the report has printed, 0.00037 below the one-column model. Q4 asks what that comparison is worth.

</details>

---

### Q4 · A fair comparison

Q3's model was fitted on seven years and the report's forecast on eight. First fit the linear regression on `fit_rows` alone and score it on the test block, as `line_fit_rmse`. Then refit `best_m` stumps on all of `train`, as the report refits every model once its settings are chosen, and score it on the test block as `boost_rmse` and on the folds as `boost_folds`. Print the three comparisons.

In [ ]:
line_fit = ...
...
line_fit_rmse = ...

boost_all = ...
...
boost_rmse = ...
boost_folds = ...

print('the same seven years:', stopped_rmse, ' against', line_fit_rmse)
print('all eight years     :', boost_rmse, ' against', one_rmse)
print('the folds           :', boost_folds, ' against', part6_one_folds)

<details>
<summary>💡 Hint 1</summary>

`LinearRegression()`, fitted on `fit_rows[['vol_20d']]` and `fit_rows['vol_next']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`GradientBoostingRegressor(n_estimators=best_m, learning_rate=0.1, max_depth=1, random_state=0)`; on the folds, `-cross_val_score(..., train[['vol_20d']], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error').mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
line_fit = LinearRegression()
line_fit.fit(fit_rows[['vol_20d']], fit_rows['vol_next'])
line_fit_rmse = rmse(test['vol_next'], line_fit.predict(test[['vol_20d']]))

boost_all = GradientBoostingRegressor(n_estimators=best_m, learning_rate=0.1, max_depth=1, random_state=0)
boost_all.fit(train[['vol_20d']], train['vol_next'])
boost_rmse = rmse(test['vol_next'], boost_all.predict(test[['vol_20d']]))
boost_folds = -cross_val_score(boost_all, train[['vol_20d']], train['vol_next'], cv=folds,
                               scoring='neg_root_mean_squared_error').mean()

print('the same seven years:', stopped_rmse, ' against', line_fit_rmse)
print('all eight years     :', boost_rmse, ' against', one_rmse)
print('the folds           :', boost_folds, ' against', part6_one_folds)
```

On the same seven years the linear regression scores 0.00390, so more than half of Q3's gap, 0.00021 of 0.00037, came from leaving out 2022, the busiest year of the training block, and the rest from the stumps. Refitted on all eight years, the stumps score 0.00398 against 0.00412. On the folds they lose, 0.00780 against 0.00755. By the rule the report has used since Part 11, a model that wins on the test block and loses on the folds has not won: the forecast stays as it is.

</details>

---

### Q5 · Twenty columns in XGBoost

Give XGBoost all twenty columns. For every depth in `[1, 2, 3]` and learning rate in `[0.3, 0.1, 0.03]`, fit up to 3,000 trees on `fit_rows`, stopped on `stop_rows` 50 rounds after the best, and store the best 2022 RMSE in a dictionary `grid_2022` and the number of trees in `trees_2022`, both keyed by `(depth, rate)`. Keep the best combination as `wide_settings`, a dictionary of `n_estimators`, `learning_rate` and `max_depth`, refit it on all of `train` as `xgb_wide`, and print its test RMSE, `xgb_rmse`, and its mean fold RMSE, `xgb_folds`, beside ridge and the tuned forest.

In [ ]:
grid_2022 = {}
trees_2022 = {}
for depth in [1, 2, 3]:
    for rate in [0.3, 0.1, 0.03]:
        ...

best = ...
wide_settings = ...
xgb_wide = ...
...
xgb_rmse = ...
xgb_folds = ...

print(best, wide_settings)
print('test :', xgb_rmse, ' ridge:', part6_ridge_rmse, ' forest:', part11_forest_rmse)
print('folds:', xgb_folds, ' ridge:', part6_ridge_folds, ' forest:', part11_forest_folds)

<details>
<summary>💡 Hint 1</summary>

Inside the loops: `model = XGBRegressor(n_estimators=3000, learning_rate=rate, max_depth=depth, early_stopping_rounds=50, random_state=0)`, fitted with `eval_set=[(stop_rows[columns], stop_rows['vol_next'])]` and `verbose=False`; then `grid_2022[(depth, rate)] = model.best_score` and `trees_2022[(depth, rate)] = model.best_iteration + 1`.

</details>

<details>
<summary>💡 Hint 2</summary>

`best = min(grid_2022, key=grid_2022.get)`, `wide_settings = {'n_estimators': trees_2022[best], 'learning_rate': best[1], 'max_depth': best[0]}`, and `XGBRegressor(random_state=0, **wide_settings)` passes the dictionary as arguments.

</details>

<details>
<summary>✅ Solution</summary>

```python
grid_2022 = {}
trees_2022 = {}
for depth in [1, 2, 3]:
    for rate in [0.3, 0.1, 0.03]:
        model = XGBRegressor(n_estimators=3000, learning_rate=rate, max_depth=depth,
                             early_stopping_rounds=50, random_state=0)
        model.fit(fit_rows[columns], fit_rows['vol_next'],
                  eval_set=[(stop_rows[columns], stop_rows['vol_next'])], verbose=False)
        grid_2022[(depth, rate)] = model.best_score
        trees_2022[(depth, rate)] = model.best_iteration + 1

best = min(grid_2022, key=grid_2022.get)
wide_settings = {'n_estimators': trees_2022[best], 'learning_rate': best[1], 'max_depth': best[0]}
xgb_wide = XGBRegressor(random_state=0, **wide_settings)
xgb_wide.fit(train[columns], train['vol_next'])
xgb_rmse = rmse(test['vol_next'], xgb_wide.predict(test[columns]))
xgb_folds = -cross_val_score(XGBRegressor(random_state=0, **wide_settings), train[columns], train['vol_next'],
                              cv=folds, scoring='neg_root_mean_squared_error').mean()

print(best, wide_settings)
print('test :', xgb_rmse, ' ridge:', part6_ridge_rmse, ' forest:', part11_forest_rmse)
print('folds:', xgb_folds, ' ridge:', part6_ridge_folds, ' forest:', part11_forest_folds)
```

Stumps at a rate of 0.3 have the lowest 2022 RMSE, 0.00577, stopped after 13 trees; every depth-1 setting beats every deeper one on 2022. Refitted on all eight years they score 0.00469 on the test block and 0.00822 on the folds: behind ridge (0.00460 and 0.00761) and the tuned forest (0.00462 and 0.00757) on both. On Apple, with twenty columns, boosting is the weakest of the three flexible forecasts.

</details>

---

### Q6 · What XGBoost leans on

Read `xgb_wide` two ways. Print its five largest `feature_importances_` and the number of columns it never cut on. Then compute its permutation importance on the test block with `scoring='neg_root_mean_squared_error'` and ten repeats, and print the three columns it loses most without and the three it does best without.

In [ ]:
gain = ...
never_cut = ...
result = ...
shuffled = ...

print(...)
print('never cut on:', never_cut)
print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`gain = pd.Series(xgb_wide.feature_importances_, index=columns).sort_values(ascending=False)`, and `(gain == 0).sum()` counts the columns never used.

</details>

<details>
<summary>💡 Hint 2</summary>

`permutation_importance(xgb_wide, test[columns], test['vol_next'], scoring='neg_root_mean_squared_error', n_repeats=10, random_state=0)`, then a sorted Series of `result.importances_mean`; `.head(3)` and `.tail(3)`. With an error as the score, a positive number means the error rose when the column was shuffled.

</details>

<details>
<summary>✅ Solution</summary>

```python
gain = pd.Series(xgb_wide.feature_importances_, index=columns).sort_values(ascending=False)
never_cut = int((gain == 0).sum())
result = permutation_importance(xgb_wide, test[columns], test['vol_next'],
                                scoring='neg_root_mean_squared_error', n_repeats=10, random_state=0)
shuffled = pd.Series(result.importances_mean, index=columns).sort_values(ascending=False)

print(gain.head(5).round(3))
print('never cut on:', never_cut)
print(shuffled.head(3).round(6))
print(shuffled.tail(3).round(6))
```

By gain, `DIS_vol` comes first with 0.310, ahead of `vol_20d` with 0.209, and 10 of the twenty columns are never cut. Shuffled on the test block, `vol_20d` matters most and `DIS_vol` least: shuffling Disney's volatility lowers the test RMSE by 0.00015. The trees found a pattern in Disney that held from 2015 to 2022 and not after; Part 11's stump on twenty columns asked about the same column first.

</details>

---

### Q7 · Three kinds of column

Shuffle groups of columns together on the test block, one permutation of the rows for the whole group, ten times each: Apple's own volatility (the six `vol_` columns), its own returns (`ret_5d`, `ret_20d`, `ret_60d` and `up_20d`), and the desk (the ten `_vol` columns of the other instruments). Do it for `xgb_wide` and for Part 6's ridge, refitted as `ridge`, and store the mean change in test RMSE of each model in a dictionary `change`, keyed by the group.

In [ ]:
groups = {
    'own volatility': ...,
    'own returns': ...,
    'the desk': ...,
}
ridge = ...
...

change = {}
for name in groups:
    ...

print(change)

<details>
<summary>💡 Hint 1</summary>

`[c for c in columns if c.startswith('vol_')]` gives the six, and `[c for c in columns if c.endswith('_vol')]` the desk. `ridge = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=part6_ridge_alpha))])`, fitted on `train[columns]`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the loop, an inner loop over k from 0 to 9: `order = np.random.default_rng(k).permutation(len(test))`, `shuffled = test.copy()`, `shuffled[groups[name]] = test[groups[name]].values[order]`, then each model's RMSE on `shuffled[columns]` minus its RMSE on `test[columns]`. Store the two means as a tuple.

</details>

<details>
<summary>✅ Solution</summary>

```python
groups = {
    'own volatility': [c for c in columns if c.startswith('vol_')],
    'own returns': ['ret_5d', 'ret_20d', 'ret_60d', 'up_20d'],
    'the desk': [c for c in columns if c.endswith('_vol')],
}
ridge = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=part6_ridge_alpha))])
ridge.fit(train[columns], train['vol_next'])

base_xgb = rmse(test['vol_next'], xgb_wide.predict(test[columns]))
base_ridge = rmse(test['vol_next'], ridge.predict(test[columns]))
change = {}
for name in groups:
    up_xgb, up_ridge = [], []
    for k in range(10):
        order = np.random.default_rng(k).permutation(len(test))
        shuffled = test.copy()
        shuffled[groups[name]] = test[groups[name]].values[order]
        up_xgb.append(rmse(test['vol_next'], xgb_wide.predict(shuffled[columns])) - base_xgb)
        up_ridge.append(rmse(test['vol_next'], ridge.predict(shuffled[columns])) - base_ridge)
    change[name] = (np.mean(up_xgb), np.mean(up_ridge))

print(pd.DataFrame(change, index=['XGBoost', 'ridge']).T)
```

Shuffling Apple's own volatility raises XGBoost's test RMSE by 0.000107 and ridge's by 0.000139. Shuffling its own returns raises XGBoost's by 0.000105 and leaves ridge's almost where it was (-0.000003). Shuffling the desk's ten columns lowers both, by 0.000085 and 0.000049: the other instruments' volatility carried patterns from 2015 to 2022 that did not hold in 2023 and 2024. That is the report's finding since Part 6, measured directly: on Apple the extra columns cost more than they bring.

</details>

---

### Q8 · The warning, once

Make the jump label for the training and test days as `train_jump` and `test_jump`. Refit Part 9's warning, a scaled logistic regression on `vol_20d`, as `warning_model`, and check its test AUC against `part9_auc`. Then boost the label: XGBoost stumps at a learning rate of 0.1, the number of trees chosen on 2022 by early stopping and then refitted on all training days, as `xgb_jump`. Collect the test and fold AUCs of the five warnings the report has fitted in a DataFrame `warning_board`.

In [ ]:
train_jump = ...
test_jump = ...
warning_model = ...
...
print('Part 9 refitted:', ...)

jump_stop = ...
...
xgb_jump = ...
...
xgb_jump_auc = ...
xgb_jump_folds = ...

warning_board = ...
print(warning_board)

<details>
<summary>💡 Hint 1</summary>

The label is `(train['vol_next'] > 1.5 * train['vol_20d']).astype(int)`. For the stopping, fit on `fit_rows[columns]` with `train_jump.loc[:'2021-12-31']` and stop on `stop_rows[columns]` with `train_jump.loc['2022-01-01':]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`warning_board = pd.DataFrame({'test': [...], 'folds': [...]}, index=[...])`, with the restored `part9_` and `part11_` numbers and your two; `.sort_values('folds')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
train_jump = (train['vol_next'] > 1.5 * train['vol_20d']).astype(int)
test_jump = (test['vol_next'] > 1.5 * test['vol_20d']).astype(int)
warning_model = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
warning_model.fit(train[['vol_20d']], train_jump)
print('Part 9 refitted:', abs(roc_auc_score(test_jump, warning_model.predict_proba(test[['vol_20d']])[:, 1])
                              - part9_auc) < 0.0001)

jump_stop = XGBClassifier(n_estimators=3000, learning_rate=0.1, max_depth=1, early_stopping_rounds=50,
                          random_state=0)
jump_stop.fit(fit_rows[columns], train_jump.loc[:'2021-12-31'],
              eval_set=[(stop_rows[columns], train_jump.loc['2022-01-01':])], verbose=False)
xgb_jump = XGBClassifier(n_estimators=jump_stop.best_iteration + 1, learning_rate=0.1, max_depth=1,
                         random_state=0)
xgb_jump.fit(train[columns], train_jump)
xgb_jump_auc = roc_auc_score(test_jump, xgb_jump.predict_proba(test[columns])[:, 1])
xgb_jump_folds = cross_val_score(xgb_jump, train[columns], train_jump, cv=folds, scoring='roc_auc').mean()

warning_board = pd.DataFrame({
    'test': [part9_auc, part9_wide_auc, part9_knn_auc, part11_jump_forest_auc, xgb_jump_auc],
    'folds': [part9_folds_auc, part9_wide_folds_auc, part9_knn_folds_auc, part11_jump_forest_folds_auc,
              xgb_jump_folds],
}, index=['logistic regression, vol_20d (Part 9)', 'logistic regression, twenty columns (Part 9)',
          'k-nearest neighbours, vol_20d (Part 9)', 'forest, twenty columns (Part 11)',
          'XGBoost, twenty columns (Part 12)'])
print(warning_board.sort_values('folds', ascending=False).round(3))
```

The refit matches Part 9: `True`. Stopped on 2022 after 116 stumps and refitted, boosting scores 0.838 on the test block and 0.705 on the folds. Of the five warnings the report has fitted, the one-column logistic regression is first on both yardsticks, 0.889 and 0.792, as it has been since Part 9; k-nearest neighbours is second on the folds, 0.759. The warning stays, threshold and all.

</details>

---

## The report as a whole

The forecast and the warning have now met every family of model in the course. The rest of this part sets them side by side: every forecast the report has fitted since Part 4 on the same rows and both yardsticks, the desk, the days that went wrong, and the report as it would go out.

---

### Q9 · Every forecast, on the same rows

Refit every forecast the report has fitted since Part 5 on `train`, with the settings the report chose for it, and score each on the test block and on the folds. The dictionary below names them, with the part each comes from and its columns; fill in the models. Then add Part 4's two rules: the training average, and persistence, which repeats `vol_20d`. On the folds, the average is the mean of each fold's fitting rows. Collect everything in a DataFrame `board`, sorted by the test RMSE, and check that it reproduces the report's numbers for ridge and the tuned forest. This takes about a minute on Colab.

In [ ]:
one, three, pair = ['vol_20d'], ['vol_20d', 'ret_20d', 'up_20d'], ['vol_20d', 'ret_20d']
candidates = {
    'linear regression, vol_20d': (5, one, ...),
    'linear regression, three columns': (5, three, ...),
    'OLS, twenty columns': (6, columns, ...),
    'ridge': (6, columns, ...),
    'lasso': (6, columns, ...),
    'elastic net': (6, columns, ...),
    'stump, vol_20d': (11, one, ...),
    'stump, twenty columns': (11, columns, ...),
    'tree of depth 2, two columns': (11, pair, ...),
    'bagging': (11, columns, ...),
    'forest': (11, columns, ...),
    'tuned forest': (11, columns, ...),
    'boosted stumps, vol_20d': (12, one, ...),
    'XGBoost, twenty columns': (12, columns, ...),
}

rows = []
for name in candidates:
    ...

...

board = ...
print(board)
print(...)

<details>
<summary>💡 Hint 1</summary>

Ridge, the lasso and the elastic net sit in a pipeline with a `StandardScaler`, with `part6_ridge_alpha`, `part6_lasso_alpha` and `part6_enet`; the tuned forest is `RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1, **part11_forest_settings)`; bagging is a forest with `max_features=1.0`, the plain forest `'sqrt'`; the boosted models use `best_m` and `wide_settings`.

</details>

<details>
<summary>💡 Hint 2</summary>

In the loop: `part, cols, model = candidates[name]`, fit, then append `{'model': name, 'part': part, 'test': ..., 'folds': ...}`. For the two rules, loop `for fit_idx, val_idx in folds.split(train):` and use `train['vol_next'].iloc[fit_idx].mean()` and `train['vol_20d'].iloc[val_idx]`. The check: `abs(board.loc['ridge', 'test'] - part6_ridge_rmse) < 0.00001`, and the same for the tuned forest.

</details>

<details>
<summary>✅ Solution</summary>

```python
one, three, pair = ['vol_20d'], ['vol_20d', 'ret_20d', 'up_20d'], ['vol_20d', 'ret_20d']
candidates = {
    'linear regression, vol_20d': (5, one, LinearRegression()),
    'linear regression, three columns': (5, three, LinearRegression()),
    'OLS, twenty columns': (6, columns, LinearRegression()),
    'ridge': (6, columns, Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=part6_ridge_alpha))])),
    'lasso': (6, columns, Pipeline([('scale', StandardScaler()),
                                    ('lasso', Lasso(alpha=part6_lasso_alpha, max_iter=20000))])),
    'elastic net': (6, columns, Pipeline([('scale', StandardScaler()),
                                          ('enet', ElasticNet(alpha=part6_enet['alpha'],
                                                              l1_ratio=part6_enet['l1_ratio'], max_iter=20000))])),
    'stump, vol_20d': (11, one, DecisionTreeRegressor(max_depth=1, random_state=0)),
    'stump, twenty columns': (11, columns, DecisionTreeRegressor(max_depth=1, random_state=0)),
    'tree of depth 2, two columns': (11, pair, DecisionTreeRegressor(max_depth=2, random_state=0)),
    'bagging': (11, columns, RandomForestRegressor(n_estimators=100, max_features=1.0, random_state=0, n_jobs=-1)),
    'forest': (11, columns, RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=0, n_jobs=-1)),
    'tuned forest': (11, columns, RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1,
                                                        **part11_forest_settings)),
    'boosted stumps, vol_20d': (12, one, GradientBoostingRegressor(n_estimators=best_m, learning_rate=0.1,
                                                                   max_depth=1, random_state=0)),
    'XGBoost, twenty columns': (12, columns, XGBRegressor(random_state=0, **wide_settings)),
}

rows = []
for name in candidates:
    part, cols, model = candidates[name]
    model.fit(train[cols], train['vol_next'])
    folds_rmse = -cross_val_score(model, train[cols], train['vol_next'], cv=folds,
                                  scoring='neg_root_mean_squared_error').mean()
    rows.append({'model': name, 'part': part, 'test': rmse(test['vol_next'], model.predict(test[cols])),
                 'folds': folds_rmse})

average_folds, persistence_folds = [], []
for fit_idx, val_idx in folds.split(train):
    actual = train['vol_next'].iloc[val_idx]
    average_folds.append(rmse(actual, np.full(len(val_idx), train['vol_next'].iloc[fit_idx].mean())))
    persistence_folds.append(rmse(actual, train['vol_20d'].iloc[val_idx]))
rows.append({'model': 'training average', 'part': 4, 'folds': np.mean(average_folds),
             'test': rmse(test['vol_next'], np.full(len(test), train['vol_next'].mean()))})
rows.append({'model': 'persistence', 'part': 4, 'folds': np.mean(persistence_folds),
             'test': rmse(test['vol_next'], test['vol_20d'])})

board = pd.DataFrame(rows).set_index('model').sort_values('test')
print(board.round(5))
print('reproduces the report:', abs(board.loc['ridge', 'test'] - part6_ridge_rmse) < 0.00001,
      abs(board.loc['tuned forest', 'test'] - part11_forest_rmse) < 0.00001)
```

Sixteen forecasts, and `True True`: one loop reproduces numbers the report printed in Parts 6 and 11. On the test block the boosted stumps lead with 0.00398, then the tree of depth 2 with 0.00411, then the report's linear regression with 0.00412. On the folds the report's linear regression leads with 0.00755, ahead of the tuned forest (0.00757) and ridge (0.00761). Both models ahead of it on the test block are behind it on the folds, by 0.00025 and 0.00062. Keep `board`: Q10 and Q11 read it.

</details>

---

### Q10 · Draw the scoreboard

Draw `board` from Q9 as two panels of horizontal bars side by side, the test block on the left and the folds on the right, with the models in the same order in both and the report's forecast in a second colour. Let the bars start at zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
...
...
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`colours = ['#b3402f' if name == 'linear regression, vol_20d' else '#1c5cab' for name in board.index]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`axes[0].barh(board.index, board['test'], color=colours)` and the same with `board['folds']` on `axes[1]`; `axes[0].invert_yaxis()` puts the best test score at the top.

</details>

<details>
<summary>✅ Solution</summary>

```python
colours = ['#b3402f' if name == 'linear regression, vol_20d' else '#1c5cab' for name in board.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
axes[0].barh(board.index, board['test'], color=colours)
axes[1].barh(board.index, board['folds'], color=colours)
axes[0].invert_yaxis()
axes[0].set_title('RMSE on the test block, 2023 and 2024', loc='left')
axes[1].set_title('mean RMSE on the five folds', loc='left')
fig.tight_layout()
plt.show()
```

On bars that start at zero most of the sixteen look alike, and they are: the best test score is 0.00013 below the report's forecast, whose error is 0.00412. The bars that stand out belong to the models that broke something: OLS on twenty columns, bagging, and the stump on twenty columns. Every bar on the right is longer than its partner on the left, because the folds include 2020 and the test years were calm.

</details>

---

### Q11 · How far the report came

Measure each forecast against the training average, the rule Part 4 started from: its skill is the share of the average's squared error it removes on the test block, $1 - (\text{RMSE} / \text{RMSE}_{\text{average}})^2$. Store it as a Series `skill`, sorted. Then use `groupby` on `part` to print the best test RMSE and the best fold RMSE of each part.

In [ ]:
average_rmse = ...
skill = ...
best_by_part = ...

print(skill)
print(best_by_part)

<details>
<summary>💡 Hint 1</summary>

`average_rmse = board.loc['training average', 'test']` and `skill = (1 - (board['test'] / average_rmse) ** 2).sort_values(ascending=False)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`board.groupby('part')[['test', 'folds']].min()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
average_rmse = board.loc['training average', 'test']
skill = (1 - (board['test'] / average_rmse) ** 2).sort_values(ascending=False)
best_by_part = board.groupby('part')[['test', 'folds']].min()

print(skill.round(3))
print(best_by_part.round(5))
```

Persistence, the rule from Part 4 that repeats this month's volatility, removes 0.386 of the average's squared error; the report's linear regression 0.402; the best model of the course 0.440. Most of the distance was covered before any model was fitted. By part, the best test RMSE went from 0.00417 (Part 4) to 0.00412 (5), up to 0.00447 with the wide models of Part 6, and back to 0.00411 (11) and 0.00398 (12). On the folds the best is still Part 5's 0.00755; no later part beat it.

</details>

---

### Q12 · Every family, across the desk

Fit XGBoost on every instrument with the depth and learning rate of `wide_settings`, the number of trees chosen on the instrument's own 2022 and refitted on its training block, and store the test RMSE in `desk_boost`. Put it in a DataFrame `desk` beside the one-column model and ridge from `part6_desk` and the forest from `part11_desk_forest`, find the family with the lowest RMSE for each instrument as `winners`, and count the wins.

In [ ]:
desk_boost = {}
for ticker in TICKERS:
    ...

desk = ...
winners = ...

print(desk)
print(winners)
print(...)

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `frame = wide_table(ticker)`, split it at the end of 2022 and the training block again at the end of 2021, early-stop an `XGBRegressor` with `max_depth=wide_settings['max_depth']` and `learning_rate=wide_settings['learning_rate']`, then refit with `best_iteration + 1` trees on the whole training block.

</details>

<details>
<summary>💡 Hint 2</summary>

`desk = pd.DataFrame({'one column': {t: part6_desk[t][1] for t in TICKERS}, 'ridge': {t: part6_desk[t][0] for t in TICKERS}, 'forest': part11_desk_forest, 'boosting': desk_boost})`; then `winners = desk.idxmin(axis=1)` and `winners.value_counts()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
desk_boost = {}
for ticker in TICKERS:
    frame = wide_table(ticker)
    tr, te = frame.loc[:'2022-12-31'], frame.loc['2023-01-01':]
    fr, sr = tr.loc[:'2021-12-31'], tr.loc['2022-01-01':]
    cols = list(frame.columns[:-1])
    stopper = XGBRegressor(n_estimators=3000, learning_rate=wide_settings['learning_rate'],
                           max_depth=wide_settings['max_depth'], early_stopping_rounds=50, random_state=0)
    stopper.fit(fr[cols], fr['vol_next'], eval_set=[(sr[cols], sr['vol_next'])], verbose=False)
    model = XGBRegressor(n_estimators=stopper.best_iteration + 1, learning_rate=wide_settings['learning_rate'],
                         max_depth=wide_settings['max_depth'], random_state=0)
    model.fit(tr[cols], tr['vol_next'])
    desk_boost[ticker] = rmse(te['vol_next'], model.predict(te[cols]))

desk = pd.DataFrame({'one column': {t: part6_desk[t][1] for t in TICKERS},
                     'ridge': {t: part6_desk[t][0] for t in TICKERS},
                     'forest': part11_desk_forest, 'boosting': desk_boost})
winners = desk.idxmin(axis=1)

print(desk.round(5))
print(winners)
print(winners.value_counts())
```

Boosting has the lowest RMSE on 4 of the 11 (DIS, JNJ, NVDA, PG), the one-column model on 3 (AAPL, KO, XOM), ridge and the forest on 2 each. No family wins the desk, and Apple is one of the instruments where the simplest one does. The spread is wide: on WMT boosting's error is twice the one column's, 0.00802 against 0.00408.

</details>

---

### Q13 · The days the forecast missed most

Compute the report's forecast error on every test day, what happened minus `one_model`'s forecast, and the warning's probability from `warning_model`. Put them in a DataFrame `days` with `vol_20d` and `vol_next`, mark whether the warning was on at `part9_threshold`, and print the five days the forecast fell furthest short and the five it overshot most.

In [ ]:
errors = ...
probability = ...
days = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint 1</summary>

`errors = test['vol_next'] - one_model.predict(test[['vol_20d']])` and `probability = warning_model.predict_proba(test[['vol_20d']])[:, 1]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`days = pd.DataFrame({'vol_20d': test['vol_20d'], 'vol_next': test['vol_next'], 'error': errors, 'probability': probability})`, then `days['warning'] = days['probability'] >= part9_threshold`. `days.nlargest(5, 'error')` and `days.nsmallest(5, 'error')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
errors = test['vol_next'] - one_model.predict(test[['vol_20d']])
probability = warning_model.predict_proba(test[['vol_20d']])[:, 1]
days = pd.DataFrame({'vol_20d': test['vol_20d'], 'vol_next': test['vol_next'],
                     'error': errors, 'probability': probability})
days['warning'] = days['probability'] >= part9_threshold

print(days.nlargest(5, 'error').round(4))
print(days.nsmallest(5, 'error').round(4))
```

The five largest shortfalls are 3 to 7 June 2024: `vol_20d` was about 0.0089 and the next twenty days ran at 0.0216. All five were jumps, and the warning was on for all five, at probabilities near 0.40. The five largest overshoots are 3 to 9 May 2024, after a busy month: `vol_20d` was about 0.0203 and 0.0089 came, and the warning was off. The forecast misses at the turns. The turns it misses upwards are the ones the warning was built for, and there it fired.

</details>

---

### Q14 · The forecast for the month after the data ends

The last day in `latest` is 31 December 2024, where the data ends. Forecast its next twenty trading days with `one_model`, and put the forecast and the day's own `vol_20d` into annualised percent, as Part 3 did: $\sigma_{\text{year}} = \sigma_{\text{day}} \times \sqrt{252}$. Compute the warning's probability for that day, and count how many of the 20 days whose target is not known yet had the warning on.

In [ ]:
last_day = ...
forecast_next = ...
annual_next = ...
annual_now = ...
probability_now = ...
unknown = ...
warning_days = ...

print('this month:', annual_now, ' next month:', annual_next)
print('warning probability:', probability_now, ' on for', warning_days, 'of the last 20 days')

<details>
<summary>💡 Hint 1</summary>

`last_day = latest.iloc[[-1]]` keeps it as a one-row table, so that `one_model.predict(last_day[['vol_20d']])[0]` works. Multiply by `np.sqrt(252)` and by 100.

</details>

<details>
<summary>💡 Hint 2</summary>

`unknown = latest[latest['vol_next'].isna()]`, and `(warning_model.predict_proba(unknown[['vol_20d']])[:, 1] >= part9_threshold).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
last_day = latest.iloc[[-1]]
forecast_next = one_model.predict(last_day[['vol_20d']])[0]
annual_next = round(100 * forecast_next * np.sqrt(252), 1)
annual_now = round(100 * last_day['vol_20d'].iloc[0] * np.sqrt(252), 1)
probability_now = round(warning_model.predict_proba(last_day[['vol_20d']])[0, 1], 3)
unknown = latest[latest['vol_next'].isna()]
warning_days = int((warning_model.predict_proba(unknown[['vol_20d']])[:, 1] >= part9_threshold).sum())

print('this month:', annual_now, ' next month:', annual_next)
print('warning probability:', probability_now, ' on for', warning_days, 'of the last 20 days')
```

December 2024 ran at 16.5 percent a year, and the forecast for the next twenty trading days is 22.2 percent: the report expects volatility to rise back towards its average. The warning's probability is 0.335, above the threshold of 0.25, and it was on for all 20 of the last days. The data ends here, so this is the one number in the report that cannot be scored yet.

</details>

---

### Q15 · Write down what you would defend

Collect what the report would publish in a dictionary `report`, and write `summarise(report)`: it prints one line per entry and ends with three verdicts, on the forecast, on the warning, and on next month.

In [ ]:
report = {
    'target': ...,
    'split': ...,
    'forecast_model': ...,
    'forecast_rmse_test': ...,
    'forecast_rmse_folds': ...,
    'forecasts_compared': ...,
    'best_on_test': ...,
    'best_on_folds': ...,
    'desk_wins': ...,
    'warning_model': ...,
    'warning_auc_test': ...,
    'warning_auc_folds': ...,
    'warning_threshold': ...,
    'next_month_percent': ...,
    'warning_now': ...,
}


def summarise(r):
    """..."""
    ...


summarise(report)

<details>
<summary>💡 Hint 1</summary>

Every value is something you already have: `part5_target`, `part11_split`, the name `'linear regression, vol_20d'`, `one_rmse`, `part6_one_folds`, `len(board)`, `board['test'].idxmin()`, `board['folds'].idxmin()`, `winners.value_counts().to_dict()`, Part 9's numbers, `annual_next`, and `'on'` or `'off'` from `probability_now`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key, value in r.items():` with an f-string, then an `if` on whether `r['best_on_folds']` is the report's forecast.

</details>

<details>
<summary>✅ Solution</summary>

```python
report = {
    'target': part5_target,
    'split': part11_split,
    'forecast_model': 'linear regression, vol_20d',
    'forecast_rmse_test': round(one_rmse, 5),
    'forecast_rmse_folds': part6_one_folds,
    'forecasts_compared': len(board),
    'best_on_test': board['test'].idxmin(),
    'best_on_folds': board['folds'].idxmin(),
    'desk_wins': winners.value_counts().to_dict(),
    'warning_model': 'logistic regression, vol_20d',
    'warning_auc_test': part9_auc,
    'warning_auc_folds': part9_folds_auc,
    'warning_threshold': part9_threshold,
    'next_month_percent': annual_next,
    'warning_now': 'on' if probability_now >= part9_threshold else 'off',
}


def summarise(r):
    """Print the report and the three verdicts it supports."""
    for key, value in r.items():
        print(f'{key:<22}{value}')
    print()
    if r['best_on_folds'] == r['forecast_model']:
        print(f"Forecast  : {r['forecast_model']} stays. It is first on the folds, and the models")
        print(f"            ahead of it on the test block, led by {r['best_on_test']}, are behind it there.")
    else:
        print(f"Forecast  : {r['best_on_folds']} is first on the folds; the report's model needs a second look.")
    print(f"Warning   : {r['warning_model']}, run at a threshold of {r['warning_threshold']}.")
    print(f"Next month: {r['next_month_percent']} percent a year, and the warning is {r['warning_now']}.")


summarise(report)
```

After twelve parts the report publishes the two models it had by Part 9, and now it can say why. Sixteen forecasts and five warnings were fitted to the same question, from two rules to boosted trees; the forecast it keeps has two coefficients and is first on the folds, and the warning has two coefficients and a threshold set from the desk's costs. The flexible models were not wasted: they are why the simple ones can be defended, and on four instruments of the desk boosting is the better forecast.

</details>

---

## 📦 What you now have

| what | where |
|:--|:--|
| the report's table, with the days still waiting for their target | `wide_table`, `table`, `latest`, `columns`, `train`, `test`, `one_model`, `one_rmse` |
| boosting on the report's column, stopped on 2022 and refitted | `stump`, `boost_one`, `fit_rows`, `stop_rows`, `stop_scores`, `best_m`, `stopped_rmse`, `line_fit_rmse`, `boost_rmse`, `boost_folds` |
| boosting on twenty columns, and what it leans on | `grid_2022`, `wide_settings`, `xgb_wide`, `xgb_rmse`, `xgb_folds`, `gain`, `shuffled`, `change` |
| the warning, refitted and boosted | `train_jump`, `test_jump`, `warning_model`, `xgb_jump`, `warning_board` |
| every forecast on the same rows | `board`, `skill`, `best_by_part` |
| the desk, every family | `desk_boost`, `desk`, `winners` |
| the misses, and next month | `days`, `annual_next`, `probability_now`, `warning_days` |
| the thing you would defend | `report` |

## What changed since Part 11

- **Boosting came closest on the report's own column.** Stopped on 2022, 46 stumps on `vol_20d` scored 0.00375, the lowest test number in the report. Fitted on the same seven years, the linear regression scores 0.00390; refitted on all eight, the stumps score 0.00398 against 0.00412, and lose on the folds, 0.00780 against 0.00755 (Q3 and Q4).
- **On twenty columns boosting came last of the three.** XGBoost stumps at a rate of 0.3, 13 trees chosen on 2022: 0.00469 and 0.00822, behind ridge and the tuned forest on both yardsticks (Q5).
- **The importance explains the wide models' trouble.** XGBoost leaned hardest on Disney's volatility, which permutation shows to hurt on the test days, and shuffling the desk's ten columns improves both XGBoost and ridge there (Q6 and Q7).
- **The warning is unchanged.** Boosted on Part 9's label it scores 0.838 and 0.705, below the one-column logistic regression on both (Q8).
- **Everything side by side.** Of sixteen forecasts, the report's is first on the folds and third on the test block (Q9), and it removes 0.402 of the average's squared error against 0.386 for Part 4's rule (Q11). Across the desk, boosting wins on 4 instruments, the one column on 3, ridge and the forest on 2 each (Q12).
- **The two halves of the report cover each other.** The forecast's five worst shortfalls were jumps in June 2024, and the warning was on for all five (Q13).

## The report, part by part

| part | the question | what it added | the number |
|:--|:--|:--|:--|
| 1 | Which of Apple and Coca-Cola moved more in 2024? | lists, slicing and a first gauge, the price range | Apple 51.22 percent, Coca-Cola 26.15 |
| 2 | Volatility from every daily return | loops, functions, a dictionary of five stocks | Nvidia 0.0330 a day, Coca-Cola 0.0080 |
| 3 | All eleven instruments | pandas: returns, annualised volatility, rankings, figures | Nvidia 53 percent a year, the S&P 500 fund 13 |
| 4 | Which names will be volatile next month? | a target, features from the past, a split by date, two baselines | the average 0.00514, persistence 0.00417 |
| 5 | The first model | linear regression on `vol_20d`, chosen on time-ordered folds | 0.00402, ahead of persistence on 11 of 11 |
| 6 | Every column the desk has | OLS, ridge, the lasso, pipelines, `GridSearchCV` | ridge 0.00460 against 0.00412; ahead on 7 of 11 |
| 8 | Will next month be busier? | logistic regression, the threshold, the AUC | AUC 0.797 |
| 9 | A jump, priced | a rare label, costs, a threshold of 0.25, calibration, k-NN | AUC 0.889; a cost of 178 against 210 |
| 11 | Trees and forests | a tree, bagging, a forest tuned on the folds, out-of-bag | the forest 0.00462, level with ridge |
| 12 | Boosting, and the report as a whole | boosting, XGBoost, importance, every model side by side | boosted stumps 0.00398 and 0.00780; the forecast stays |

Parts 1 to 3 described a year that had happened. Part 4 turned the report into a question about the next month and set the rules every later part kept: a target that cannot see the future, a split by date, and baselines to beat. Parts 5 to 12 then brought one family of models after another to that question, and each was held to the same test block and the same folds.

## Where this leaves the risk report

The report ends where Part 5 left its forecast and Part 9 its warning, and that is a result, not a failure to improve. On Apple, the volatility of the last twenty days, used with two coefficients, carries almost everything that the desk's twenty columns and every family of model could find. Across the desk the answer differs by instrument, and the report says so.

Part 4 named three problems with its own setup. Two have answers. One split is one experiment, so every model was held to five time-ordered folds as well. The rows are not independent, which is why the folds keep time in order and why out-of-bag flattered the forest. The third, that the regimes differ, has no answer: every forecast fitted on 2015 to 2022 ran high, on average, in the calm years that followed, and the jump warning is there for the turns that no forecast sees coming.

**This is the last part of the case.** The report is complete: a forecast, a warning, the evidence for both, and a forecast for the month after the data ends.